In [6]:
# Cell 1: portable project setup
import importlib
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / ".env").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
        nested_classifier = candidate / "classifier"
        if (
            (nested_classifier / ".env").is_file()
            and (nested_classifier / "pyproject.toml").is_file()
        ):
            return nested_classifier
    raise FileNotFoundError(
        "Не найден корень classificator: нужна папка с .env и pyproject.toml"
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SOURCE_ROOT = PROJECT_ROOT / "src"
os.chdir(PROJECT_ROOT)
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

BATCH_SIZE = 30
BATCH_INDEX = 64  # starts at cases [30:60]; use 0 for [0:30]
MAX_CASES = 2000
CASE_OFFSET = BATCH_INDEX * BATCH_SIZE
MAX_WORKERS = 4
REBUILD_MANIFEST = False
AUTO_START_BATCH_INDEX = BATCH_INDEX
AUTO_BATCH_COUNT = None  # None = process all batches up to MAX_CASES
BATCH_TAG = f"batch_{BATCH_INDEX:04d}"
print("project root:", PROJECT_ROOT)
print(
    "batch:",
    BATCH_TAG,
    "offset:",
    CASE_OFFSET,
    "size:",
    BATCH_SIZE,
    "workers:",
    MAX_WORKERS,
    "auto start:",
    AUTO_START_BATCH_INDEX,
    "auto count:",
    AUTO_BATCH_COUNT,
)


project root: /Users/zero/GITHUB/vue-graph-rag/classifier
batch: batch_0064 offset: 1920 size: 30 workers: 4 auto start: 64 auto count: None


In [7]:
# Cell 2: importlib для каждого production-модуля
import importlib


def load_module(name: str, reload: bool = True):
    module = importlib.import_module(name)
    if reload:
        module = importlib.reload(module)
    print(f"{name:45} -> {module.__file__}")
    return module


config = load_module("telegram_classificator.conf", reload=True)
config.configure_logging()
qa_parser = load_module("telegram_classificator.qa_parser", reload=True)
promt = load_module("telegram_classificator.promt", reload=True)
agent = load_module("telegram_classificator.agent", reload=True)
onboarding_agent = load_module(
    "telegram_classificator.onboarding_agent", reload=True
)
print(config.summary())


telegram_classificator.conf                   -> /Users/zero/GITHUB/vue-graph-rag/classifier/src/telegram_classificator/conf.py
telegram_classificator.qa_parser              -> /Users/zero/GITHUB/vue-graph-rag/classifier/src/telegram_classificator/qa_parser.py
telegram_classificator.promt                  -> /Users/zero/GITHUB/vue-graph-rag/classifier/src/telegram_classificator/promt.py
telegram_classificator.agent                  -> /Users/zero/GITHUB/vue-graph-rag/classifier/src/telegram_classificator/agent.py
telegram_classificator.onboarding_agent       -> /Users/zero/GITHUB/vue-graph-rag/classifier/src/telegram_classificator/onboarding_agent.py
{'PROJECT_ROOT': '/Users/zero/GITHUB/vue-graph-rag/classifier', 'QA_JSON': '/Users/zero/GITHUB/vue-graph-rag/classifier/qa.json', 'IMAGE_DIR': '/Users/zero/GITHUB/vue-graph-rag/classifier', 'DATA_DIR': '/Users/zero/GITHUB/vue-graph-rag/classifier/data', 'TEXT_MODEL': 'gemma4:latest', 'VISION_MODEL': 'qwen3-vl:8b', 'VISION_BASE_URL': 'http:

In [8]:
# Cell 3: deterministic manifest and one batch of local qa.json
import json


def discover_qa_files(input_path: Path) -> list[Path]:
    if input_path.is_file():
        return [input_path]
    if input_path.is_dir():
        return sorted(input_path.rglob("qa.json"))
    raise FileNotFoundError(f"QA path not found: {input_path}")


qa_input = config.QA_JSON
if not qa_input.exists():
    qa_input = config.DATA_DIR
    print(
        "configured QA_JSON was not found; discovering qa.json under:",
        qa_input,
    )

qa_files = discover_qa_files(qa_input)
if not qa_files:
    raise FileNotFoundError(f"No qa.json files found under: {qa_input}")

MASTER_CASES = config.DATA_DIR / "cases_manifest.jsonl"
if MASTER_CASES.exists() and not REBUILD_MANIFEST:
    all_cases = [
        json.loads(line)
        for line in MASTER_CASES.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print("loaded master manifest:", MASTER_CASES)
    print("manifest total:", len(all_cases))
else:
    all_cases = []
    for qa_file in qa_files:
        print(f"parsing {qa_file.parent.name}/qa.json")
        parsed_cases = qa_parser.qa_to_cases(
            qa_file,
            case_id_prefix=qa_file.parent.name,
        )
        all_cases.extend(parsed_cases)
        print("parsed cases:", len(parsed_cases), "manifest total:", len(all_cases))
    qa_parser.write_jsonl(all_cases, MASTER_CASES)
    print("master manifest written:", MASTER_CASES)

batch_start = CASE_OFFSET
processing_limit = min(MAX_CASES, len(all_cases))
batch_end = min(CASE_OFFSET + BATCH_SIZE, processing_limit)
cases = all_cases[batch_start:batch_end]
if not cases and all_cases:
    raise IndexError(
        f"batch offset {CASE_OFFSET} is outside processing range 0:{processing_limit}"
    )

BATCH_CASES = config.DATA_DIR / f"cases_{BATCH_TAG}.jsonl"
BATCH_RESULTS = config.DATA_DIR / f"results_{BATCH_TAG}.jsonl"
BATCH_CLASS_ERRORS = config.DATA_DIR / f"classification_errors_{BATCH_TAG}.jsonl"
BATCH_ONBOARDING_RESULTS = (
    config.DATA_DIR / f"onboarding_results_{BATCH_TAG}.jsonl"
)
BATCH_ONBOARDING_ERRORS = (
    config.DATA_DIR / f"onboarding_errors_{BATCH_TAG}.jsonl"
)
BATCH_ERRORS = config.DATA_DIR / f"errors_{BATCH_TAG}.jsonl"
BATCH_MANIFEST = config.DATA_DIR / f"manifest_{BATCH_TAG}.json"

qa_parser.write_jsonl(cases, BATCH_CASES)
manifest = {
    "status": "parsed",
    "batch_index": BATCH_INDEX,
    "batch_tag": BATCH_TAG,
    "batch_size": BATCH_SIZE,
    "max_cases": MAX_CASES,
    "case_offset": CASE_OFFSET,
    "total_cases": len(all_cases),
    "processable_cases": processing_limit,
    "selected_cases": len(cases),
    "remaining_cases": max(0, processing_limit - batch_end),
    "qa_files_discovered": len(qa_files),
    "image_description_source": "qa_json",
    "artifact_paths": {
        "master_cases": str(MASTER_CASES),
        "cases": str(BATCH_CASES),
        "results": str(BATCH_RESULTS),
        "classification_errors": str(BATCH_CLASS_ERRORS),
        "onboarding_results": str(BATCH_ONBOARDING_RESULTS),
        "onboarding_errors": str(BATCH_ONBOARDING_ERRORS),
        "errors": str(BATCH_ERRORS),
    },
}
BATCH_MANIFEST.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("qa input:", qa_input)
print("qa files discovered:", len(qa_files))
print("manifest total:", len(all_cases))
print("processing limit:", processing_limit)
print("batch selected:", len(cases), f"[{CASE_OFFSET}:{batch_end}]")
print("cases output:", BATCH_CASES)
print("manifest output:", BATCH_MANIFEST)


configured QA_JSON was not found; discovering qa.json under: /Users/zero/GITHUB/vue-graph-rag/classifier/data
loaded master manifest: /Users/zero/GITHUB/vue-graph-rag/classifier/data/cases_manifest.jsonl
manifest total: 1944
qa input: /Users/zero/GITHUB/vue-graph-rag/classifier/data
qa files discovered: 132
manifest total: 1944
processing limit: 1944
batch selected: 24 [1920:1944]
cases output: /Users/zero/GITHUB/vue-graph-rag/classifier/data/cases_batch_0064.jsonl
manifest output: /Users/zero/GITHUB/vue-graph-rag/classifier/data/manifest_batch_0064.json


In [4]:
# Cell 4: use image descriptions already stored in qa.json
IMAGE_DESCRIPTION_FIELDS = ("caption", "visible_text", "visual_summary")
image_occurrences = [image for case in cases for image in case.get("images", [])]
described_image_occurrences = sum(
    1
    for image in image_occurrences
    if any(image.get(field) for field in IMAGE_DESCRIPTION_FIELDS)
)
missing_image_descriptions = len(image_occurrences) - described_image_occurrences
print(
    "qa.json image descriptions:",
    described_image_occurrences,
    "of",
    len(image_occurrences),
    "image occurrences; missing:",
    missing_image_descriptions,
)

shown_images = 0
for case in cases:
    for image in case.get("images", []):
        print("case:", case.get("case_id"))
        print("path:", image.get("relative_path"))
        print("caption:", image.get("caption"))
        print("visible_text:", image.get("visible_text"))
        print("visual_summary:", image.get("visual_summary"))
        print("---")
        shown_images += 1
        if shown_images >= 10:
            break
    if shown_images >= 10:
        break
print("image_describer disabled; classifiers use qa.json attachment descriptions")


qa.json image descriptions: 47 of 47 image occurrences; missing: 0
case: maria_andreeva__wariiandr__q_0015
path: images/veu_support/-1003510906840/574619648.jpg
caption: 
visible_text: Setup guide requests two new Russian numbers and one foreign number and explains their purposes.
visual_summary: First page of the account and phone-number setup instructions.
---
case: maria_andreeva__wariiandr__q_0015
path: images/veu_support/-1003510906840/575668224.jpg
caption: 
visible_text: Guide explains how to create two new Gmail addresses for the two HH accounts.
visual_summary: Continuation of setup instructions for foreign-market HH accounts.
---
case: maria_andreeva__wariiandr__q_0015
path: images/veu_support/-1003510906840/576716800.jpg
caption: 
visible_text: Guide explains registering HH and Telegram accounts with the newly created numbers.
visual_summary: Final page of setup instructions for the separate job-search accounts.
---
case: maria_andreeva__wariiandr__q_0018
path: images/veu_su

In [9]:
# Cell 5: classification, onboarding classification, errors and manifest

def read_jsonl_rows(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


try:
    agent.run(
        BATCH_CASES,
        BATCH_RESULTS,
        img_desc_path=None,
        limit=None,
        max_workers=MAX_WORKERS,
        errors_path=BATCH_CLASS_ERRORS,
    )
except Exception as error:
    manifest["status"] = "failed"
    manifest["failed_stage"] = "classification"
    manifest["failure"] = str(error)[:1000]
    BATCH_MANIFEST.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    raise

try:
    onboarding_agent.run(
        BATCH_CASES,
        BATCH_ONBOARDING_RESULTS,
        img_desc_path=None,
        limit=None,
        max_workers=MAX_WORKERS,
        errors_path=BATCH_ONBOARDING_ERRORS,
    )
except Exception as error:
    manifest["status"] = "failed"
    manifest["failed_stage"] = "onboarding_classification"
    manifest["failure"] = str(error)[:1000]
    BATCH_MANIFEST.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    raise

classification_error_rows = read_jsonl_rows(BATCH_CLASS_ERRORS)
onboarding_error_rows = read_jsonl_rows(BATCH_ONBOARDING_ERRORS)
all_error_rows = classification_error_rows + onboarding_error_rows
with BATCH_ERRORS.open("w", encoding="utf-8") as error_file:
    for row in all_error_rows:
        error_file.write(json.dumps(row, ensure_ascii=False) + "\n")

result_rows = read_jsonl_rows(BATCH_RESULTS)
onboarding_rows = read_jsonl_rows(BATCH_ONBOARDING_RESULTS)
manifest.update(
    {
        "status": "completed",
        "successful_cases": sum(1 for row in result_rows if not row.get("error")),
        "onboarding_successful_cases": sum(
            1 for row in onboarding_rows if not row.get("error")
        ),
        "classification_errors": len(classification_error_rows),
        "onboarding_errors": len(onboarding_error_rows),
        "image_description_source": "qa_json",
        "total_errors": len(all_error_rows),
    }
)
BATCH_MANIFEST.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("classification output:", BATCH_RESULTS)
print("onboarding output:", BATCH_ONBOARDING_RESULTS)
print("classification errors:", BATCH_CLASS_ERRORS)
print("onboarding errors:", BATCH_ONBOARDING_ERRORS)
print("all errors:", BATCH_ERRORS)
print("batch manifest:", BATCH_MANIFEST)
print(
    "batch summary:",
    "classification_success=", manifest["successful_cases"],
    "onboarding_success=", manifest["onboarding_successful_cases"],
    "errors=", manifest["total_errors"],
    "remaining=", manifest["remaining_cases"],
)


2026-11-15 22:43:39,128 INFO telegram_classificator.agent start cases=24 model=gemma4:latest workers=4
2026-11-15 22:43:39,131 INFO telegram_classificator.agent classification stage started cases=24 workers=4
2026-11-15 22:43:39,132 INFO telegram_classificator.agent case started case=yuriy_mishin__godevman__q_0002 model=gemma4:latest
2026-11-15 22:43:39,132 INFO telegram_classificator.agent case started case=yuriy_mishin__godevman__q_0003 model=gemma4:latest
2026-11-15 22:43:39,133 INFO telegram_classificator.agent case started case=yuriy_mishin__godevman__q_0004 model=gemma4:latest
2026-11-15 22:43:39,134 INFO telegram_classificator.agent case started case=yuriy_mishin__godevman__q_0005 model=gemma4:latest
classify:   0%|          | 0/24 [00:00<?, ?case/s]2026-11-15 22:44:02,015 INFO telegram_classificator.agent case yuriy_mishin__godevman__q_0004 ok, questions=1
2026-11-15 22:44:02,018 INFO telegram_classificator.agent case started case=yuriy_mishin__godevman__q_0006 model=gemma4:lat

classification output: /Users/zero/GITHUB/vue-graph-rag/classifier/data/results_batch_0064.jsonl
onboarding output: /Users/zero/GITHUB/vue-graph-rag/classifier/data/onboarding_results_batch_0064.jsonl
classification errors: /Users/zero/GITHUB/vue-graph-rag/classifier/data/classification_errors_batch_0064.jsonl
onboarding errors: /Users/zero/GITHUB/vue-graph-rag/classifier/data/onboarding_errors_batch_0064.jsonl
all errors: /Users/zero/GITHUB/vue-graph-rag/classifier/data/errors_batch_0064.jsonl
batch manifest: /Users/zero/GITHUB/vue-graph-rag/classifier/data/manifest_batch_0064.json
batch summary: classification_success= 24 onboarding_success= 22 errors= 2 remaining= 0


In [5]:
# Cell 6: automatically process missing batches
from tqdm.auto import tqdm


def _read_jsonl_rows(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def _batch_artifacts(batch_index: int) -> dict[str, Path]:
    tag = f"batch_{batch_index:04d}"
    return {
        "cases": config.DATA_DIR / f"cases_{tag}.jsonl",
        "results": config.DATA_DIR / f"results_{tag}.jsonl",
        "classification_errors": config.DATA_DIR / f"classification_errors_{tag}.jsonl",
        "onboarding_results": config.DATA_DIR / f"onboarding_results_{tag}.jsonl",
        "onboarding_errors": config.DATA_DIR / f"onboarding_errors_{tag}.jsonl",
        "errors": config.DATA_DIR / f"errors_{tag}.jsonl",
        "manifest": config.DATA_DIR / f"manifest_{tag}.json",
    }


def _write_batch_manifest(manifest: dict, path: Path) -> None:
    path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def _process_batch(batch_index: int) -> dict:
    artifacts = _batch_artifacts(batch_index)
    batch_start = batch_index * BATCH_SIZE
    processing_limit = min(MAX_CASES, len(all_cases))
    batch_end = min(batch_start + BATCH_SIZE, processing_limit)
    cases_for_batch = all_cases[batch_start:batch_end]
    if not cases_for_batch:
        raise IndexError(
            f"batch index {batch_index} is outside processing range 0:{processing_limit}"
        )

    manifest_path = artifacts["manifest"]
    if manifest_path.exists():
        existing_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        required_outputs = tuple(
            artifacts[name]
            for name in (
                "cases",
                "results",
                "classification_errors",
                "onboarding_results",
                "onboarding_errors",
                "errors",
            )
        )
        if existing_manifest.get("status") == "completed" and all(
            path.exists() for path in required_outputs
        ):
            print(
                f"skip completed batch={batch_index} "
                f"range=[{batch_start}:{batch_end}]"
            )
            return existing_manifest

    manifest = {
        "status": "parsed",
        "batch_index": batch_index,
        "batch_tag": f"batch_{batch_index:04d}",
        "batch_size": BATCH_SIZE,
        "max_cases": MAX_CASES,
        "case_offset": batch_start,
        "total_cases": len(all_cases),
        "processable_cases": processing_limit,
        "selected_cases": len(cases_for_batch),
        "remaining_cases": max(0, processing_limit - batch_end),
        "artifact_paths": {
            "master_cases": str(MASTER_CASES),
            **{name: str(path) for name, path in artifacts.items()},
        },
    }
    qa_parser.write_jsonl(cases_for_batch, artifacts["cases"])
    _write_batch_manifest(manifest, manifest_path)
    print(
        f"start batch={batch_index} "
        f"range=[{batch_start}:{batch_end}] cases={len(cases_for_batch)}"
    )

    stored_image_count = sum(
        1
        for case in cases_for_batch
        for image in case.get("images", [])
        if any(image.get(field) for field in IMAGE_DESCRIPTION_FIELDS)
    )
    manifest["image_description_source"] = "qa_json"
    print(
        f"use qa.json image descriptions batch={batch_index} "
        f"described_occurrences={stored_image_count}"
    )

    try:
        agent.run(
            artifacts["cases"],
            artifacts["results"],
            img_desc_path=None,
            limit=None,
            max_workers=MAX_WORKERS,
            errors_path=artifacts["classification_errors"],
        )
    except Exception as error:
        manifest.update(
            {
                "status": "failed",
                "failed_stage": "classification",
                "failure": str(error)[:1000],
            }
        )
        _write_batch_manifest(manifest, manifest_path)
        raise

    try:
        onboarding_agent.run(
            artifacts["cases"],
            artifacts["onboarding_results"],
            img_desc_path=None,
            limit=None,
            max_workers=MAX_WORKERS,
            errors_path=artifacts["onboarding_errors"],
        )
    except Exception as error:
        manifest.update(
            {
                "status": "failed",
                "failed_stage": "onboarding_classification",
                "failure": str(error)[:1000],
            }
        )
        _write_batch_manifest(manifest, manifest_path)
        raise

    classification_errors = _read_jsonl_rows(artifacts["classification_errors"])
    onboarding_errors = _read_jsonl_rows(artifacts["onboarding_errors"])
    all_errors = classification_errors + onboarding_errors
    with artifacts["errors"].open("w", encoding="utf-8") as error_file:
        for row in all_errors:
            error_file.write(json.dumps(row, ensure_ascii=False) + "\n")

    result_rows = _read_jsonl_rows(artifacts["results"])
    onboarding_rows = _read_jsonl_rows(artifacts["onboarding_results"])
    manifest.update(
        {
            "status": "completed",
            "successful_cases": sum(
                1 for row in result_rows if not row.get("error")
            ),
            "onboarding_successful_cases": sum(
                1 for row in onboarding_rows if not row.get("error")
            ),
            "classification_errors": len(classification_errors),
            "onboarding_errors": len(onboarding_errors),
            "image_description_source": "qa_json",
            "total_errors": len(all_errors),
        }
    )
    _write_batch_manifest(manifest, manifest_path)
    print(
        f"completed batch={batch_index} "
        f"success={manifest['successful_cases']} "
        f"errors={manifest['total_errors']}"
    )
    return manifest


if AUTO_START_BATCH_INDEX < 0:
    raise ValueError("AUTO_START_BATCH_INDEX must be non-negative")
if AUTO_BATCH_COUNT is not None and AUTO_BATCH_COUNT < 0:
    raise ValueError("AUTO_BATCH_COUNT must be non-negative or None")

last_batch_index = (min(MAX_CASES, len(all_cases)) - 1) // BATCH_SIZE
if AUTO_BATCH_COUNT is None:
    auto_end_batch = last_batch_index + 1
else:
    auto_end_batch = min(
        AUTO_START_BATCH_INDEX + AUTO_BATCH_COUNT,
        last_batch_index + 1,
    )
if AUTO_START_BATCH_INDEX > auto_end_batch:
    raise ValueError("AUTO_START_BATCH_INDEX is outside the selected batch range")

batch_indexes = range(AUTO_START_BATCH_INDEX, auto_end_batch)
print(
    "auto batch range:",
    f"[{AUTO_START_BATCH_INDEX}:{auto_end_batch})",
    "case range:",
    f"[{AUTO_START_BATCH_INDEX * BATCH_SIZE}:{min(auto_end_batch * BATCH_SIZE, MAX_CASES)}]",
)
for batch_index in tqdm(batch_indexes, desc="auto batches", unit="batch"):
    _process_batch(batch_index)
print("automatic batch processing finished")


auto batch range: [38:65) case range: [1140:1950]


auto batches:   0%|          | 0/27 [00:00<?, ?batch/s]2026-11-14 19:42:37,867 INFO telegram_classificator.agent start cases=30 model=gemma4:latest workers=4
2026-11-14 19:42:37,868 INFO telegram_classificator.agent classification stage started cases=30 workers=4
2026-11-14 19:42:37,868 INFO telegram_classificator.agent case started case=maria_andreeva__wariiandr__q_0009 model=gemma4:latest


start batch=38 range=[1140:1170] cases=30
use qa.json image descriptions batch=38 described_occurrences=47


2026-11-14 19:42:37,869 INFO telegram_classificator.agent case started case=maria_andreeva__wariiandr__q_0010 model=gemma4:latest

classify:   0%|          | 0/30 [00:00<?, ?case/s]2026-11-14 19:42:37,869 INFO telegram_classificator.agent case started case=maria_andreeva__wariiandr__q_0011 model=gemma4:latest
2026-11-14 19:42:37,869 INFO telegram_classificator.agent case started case=maria_andreeva__wariiandr__q_0012 model=gemma4:latest
2026-11-14 19:42:45,898 INFO telegram_classificator.agent case maria_andreeva__wariiandr__q_0009 ok, questions=1
2026-11-14 19:42:45,899 INFO telegram_classificator.agent case started case=maria_andreeva__wariiandr__q_0013 model=gemma4:latest

classify:   3%|▎         | 1/30 [00:08<03:52,  8.03s/case]2026-11-14 19:42:46,049 INFO telegram_classificator.agent case maria_andreeva__wariiandr__q_0010 ok, questions=1
2026-11-14 19:42:46,050 INFO telegram_classificator.agent case started case=maria_andreeva__wariiandr__q_0014 model=gemma4:latest

classify:   7

completed batch=38 success=30 errors=1
start batch=39 range=[1170:1200] cases=30
use qa.json image descriptions batch=39 described_occurrences=71


2026-11-14 19:44:55,828 INFO telegram_classificator.agent case started case=mark_tsarkov__unrealcapabilities__q_0004 model=gemma4:latest

classify:   0%|          | 0/30 [00:00<?, ?case/s]2026-11-14 19:44:55,829 INFO telegram_classificator.agent case started case=mark_tsarkov__unrealcapabilities__q_0005 model=gemma4:latest
2026-11-14 19:45:11,916 INFO telegram_classificator.agent case mark_tsarkov__unrealcapabilities__q_0003 ok, questions=1
2026-11-14 19:45:11,917 INFO telegram_classificator.agent case started case=mark_tsarkov__unrealcapabilities__q_0006 model=gemma4:latest

classify:   3%|▎         | 1/30 [00:16<07:46, 16.08s/case]2026-11-14 19:45:13,552 INFO telegram_classificator.agent case mark_tsarkov__unrealcapabilities__q_0002 ok, questions=1
2026-11-14 19:45:13,552 INFO telegram_classificator.agent case started case=mark_tsarkov__unrealcapabilities__q_0007 model=gemma4:latest

classify:   7%|▋         | 2/30 [00:17<03:32,  7.58s/case]2026-11-14 19:45:14,476 INFO telegram_class

... (47 outputs left out)

2026-11-15 18:42:58,950 INFO telegram_classificator.agent case started case=vsevolod_nasonov__imagine2509__q_0028 model=gemma4:latest

classify:   0%|          | 0/30 [00:00<?, ?case/s]2026-11-15 18:43:08,354 INFO telegram_classificator.agent case vsevolod_nasonov__imagine2509__q_0026 ok, questions=1
2026-11-15 18:43:08,354 INFO telegram_classificator.agent case started case=vsevolod_nasonov__imagine2509__q_0029 model=gemma4:latest

classify:   3%|▎         | 1/30 [00:09<04:32,  9.40s/case]2026-11-15 18:43:08,718 INFO telegram_classificator.agent case vsevolod_nasonov__imagine2509__q_0025 ok, questions=1
2026-11-15 18:43:08,719 INFO telegram_classificator.agent case started case=vsevolod_nasonov__imagine2509__q_0030 model=gemma4:latest

classify:   7%|▋         | 2/30 [00:09<01:54,  4.08s/case]2026-11-15 18:43:10,156 INFO telegram_classificator.agent case vsevolod_nasonov__imagine2509__q_0027 ok, questions=1
2026-11-15 18:43:10,157 INFO telegram_classificator.agent case started case=vs

completed batch=63 success=30 errors=1
start batch=64 range=[1920:1944] cases=24
use qa.json image descriptions batch=64 described_occurrences=7


2026-11-15 18:45:34,725 INFO telegram_classificator.agent case started case=yuriy_mishin__godevman__q_0005 model=gemma4:latest

classify:   0%|          | 0/24 [00:00<?, ?case/s]2026-11-15 18:45:43,342 INFO telegram_classificator.agent case yuriy_mishin__godevman__q_0004 ok, questions=1
2026-11-15 18:45:43,343 INFO telegram_classificator.agent case started case=yuriy_mishin__godevman__q_0006 model=gemma4:latest

classify:   4%|▍         | 1/24 [00:08<03:18,  8.61s/case]2026-11-15 18:45:43,848 INFO telegram_classificator.agent case yuriy_mishin__godevman__q_0005 ok, questions=1
2026-11-15 18:45:43,849 INFO telegram_classificator.agent case started case=yuriy_mishin__godevman__q_0007 model=gemma4:latest

classify:   8%|▊         | 2/24 [00:09<01:24,  3.84s/case]2026-11-15 18:45:44,767 INFO telegram_classificator.agent case yuriy_mishin__godevman__q_0002 ok, questions=1
2026-11-15 18:45:44,768 INFO telegram_classificator.agent case started case=yuriy_mishin__godevman__q_0008 model=gemma4:

completed batch=64 success=24 errors=4
automatic batch processing finished
